# LiteViz Tutorial

Welcome to the LiteViz tutorial. This notebook demonstrates how to use the LiteViz library, from basic image viewing to building custom interactive tools.

### Installation
For Google Colab, you'll need to install `ipyevents` (and ensure custom widgets are enabled). `ipywidgets`, `scipy` and `Pillow` are usually pre-installed. For local environments, just run the pip install.

In [ ]:
!pip install ipyevents ipywidgets Pillow scipy

# For Colab, you might need to enable custom widget manager:
# from google.colab import output
# output.enable_custom_widget_manager()

### 1. Generating Sample Data
Let's generate a synthetic 3D volume (like a medical phantom) and a dummy mask so we have something to visualize.

In [ ]:
import numpy as np
import scipy.ndimage

In [ ]:
def generate_phantom():
    print("Generating synthetic 3D phantom...")
    D, H, W = 60, 256, 256
    base_noise = np.random.normal(0, 1, (D, H, W)).astype(np.float32)
    structures = scipy.ndimage.gaussian_filter(base_noise, sigma=[4, 15, 15])
    structures = (structures - structures.min()) / (structures.max() - structures.min())
    
    clean_image = (structures ** 1.5 * 3000) - 1000
    mask = np.where(clean_image > 130, 1, 0).astype(np.uint16)
    
    texture = scipy.ndimage.gaussian_filter(np.random.normal(0, 1, (D, H, W)).astype(np.float32), sigma=1.0)
    image = np.clip(clean_image + (texture * 100), -32768, 32767).astype(np.int16)
    return image, mask

image, mask = generate_phantom()
print(f"Image shape: {image.shape}, Mask shape: {mask.shape}")

In [ ]:
data = np.load('../../Saros/example.npz')
image,mask = data['image'],data['mask']

### 2. Basic Slider-Driven Viewer (`DicomWidget`)
The simplest way to view a 3D image is using `DicomWidget`. It provides basic UI sliders and does not require `ipyevents` (safe for all environments).

In [ ]:
# Add LiteViz to path if running from repo root or clone
import sys, os
if os.path.abspath('..') not in sys.path:
    sys.path.insert(0, os.path.abspath('..'))

from dicom_utils import DicomWidget

# Initialize and display the simple widget
basic_widget = DicomWidget(image_array=image, mask=mask)
basic_widget.display()

### 3. Advanced Mouse-Driven Viewer (`InteractiveSlicer`)
The `InteractiveSlicer` (or `InteractiveDicomWidget`) uses `ipyevents` to provide a professional radiology-style experience. You can scroll through slices with the mouse wheel and adjust Window/Level by right-clicking and dragging.

In [ ]:
from dicom_utils import InteractiveSlicer

app = InteractiveSlicer(
    image_array=image, 
    mask=mask,
    fps=20,             # Target frames per second
    show_status=True    # Show the status bar with HU and slice info
)

app.display()

### 4. Custom Extension: Intercepting Callbacks
Because LiteViz is modular, we can access the underlying viewer directly and attach our own custom callbacks. Let's create a tool that prints the coordinates whenever you click on the image.

In [ ]:
from dicom_utils import InteractiveSlicer
import ipywidgets as widgets
from IPython.display import display

# Create the standard interactive app
custom_app = InteractiveSlicer(image_array=image, mask=mask)

# Create an output widget to capture prints
out = widgets.Output(layout={'border': '1px solid black', 'height': '100px', 'overflow': 'auto'})

# Define a custom click handler
def my_click_handler(x, y, button):
    with out:
        # Get the current slice index and HU value
        z = custom_app.slicer.state['z_index']
        val = custom_app.slicer.get_value_at_jk(y, x)
        print(f"Clicked at (x={x}, y={y}, z={z}) - Button {button} - Value: {val}")

# Attach our custom handler to the viewer's on_click event
custom_app.viewer.on_click = my_click_handler

# Display the app and our custom output log together
display(widgets.VBox([custom_app.widget, out]))

### 5. Drawing and Annotation (`AnnotationCanvas`)
The `AnnotationCanvas` wraps a base widget to provide advanced drawing and click interaction tools. 

**Interface Requirements for `base_w`:**
To be used as the `base_w` argument, a widget must expose:
- `.im_w`: An `ipywidgets.Image` (or similar) which is the DOM event source.
- `.slicer`: An object managing state, including `.mask` (a NumPy array) and `.state['z_index']`.
- `._update_image(...)`: A method to refresh the visual display after mask changes.

**Interactions:**
- **Edit Mask:** Hold **`Command` (Mac) or `Ctrl` (Windows/Linux)** while left-clicking and dragging.
- **Window/Level (HU):** Right-click and drag.
- **Fast Scrolling:** Mouse wheel (unthrottled).
- **Click Callbacks:** If `edit_flag=True` and a callback is provided, it triggers on **`Command/Ctrl` + Click**.

In [ ]:
from dicom_utils import DicomWidget, AnnotationCanvas

# We need a DicomWidget as the base
base_w = DicomWidget(image_array=image, mask=mask)

# Wrap it in an AnnotationCanvas (edit_flag=True enables painting/clicking via Cmd/Ctrl modifier)
canvas = AnnotationCanvas(base_w, edit_flag=True)

canvas.display()

### 6. Using `AnnotationCanvas` as a Lab for Custom Objects

You can use `AnnotationCanvas` to build a "lab" for experimenting with much larger or more complex data objects. By separating the UI interaction (handled by the canvas) from the underlying object (represented by a widget following the `SimpleRGBWidget` interface), you can easily test callbacks and real-time updates.

In [ ]:
from dicom_utils import SimpleRGBWidget, AnnotationCanvas
import numpy as np

# 1. Create a simple RGB image (e.g., 256x256 with a gradient)
x = np.linspace(0, 255, 256, dtype=np.uint8)
y = np.linspace(0, 255, 256, dtype=np.uint8)
X, Y = np.meshgrid(x, y)
rgb_array = np.stack([X, Y, np.full_like(X, 128)], axis=-1)

# 2. Initialize our "lab" widget with this RGB data
lab_widget = SimpleRGBWidget(rgb_array=rgb_array)

# 3. Wrap it in a canvas with a custom callback
def lab_callback(x, y, z):
    print(f"Experiment Triggered: Lab Object updated at {x=}, {y=}, {z=}")

canvas_lab = AnnotationCanvas(lab_widget, edit_flag=True, on_click_callback=lab_callback)

# 4. Display the lab. Remember to hold Cmd/Ctrl to paint or trigger the callback!
canvas_lab.display()